# Customer Churn Prediction with XGBoost

**Author:** Olivier Robert-Duboille

## 1. Introduction
Customer churn (attrition) is a critical metric for any subscription-based business. In this notebook, we leverage XGBoost, a state-of-the-art gradient boosting algorithm, to predict which customers are at risk of leaving.

### Objectives:
- Simulate a telecom customer dataset with demographics, usage patterns, and contract details.
- Analyze feature distributions and target imbalance.
- Build an XGBoost classifier.
- Evaluate performance using ROC-AUC and Confusion Matrix.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Data Simulation
We generate 2000 customers. 
- **Features:** `ContractType` (Month-to-month is riskier), `MonthlyCharges`, `Tenure` (New customers churn more), `TotalUsage`.
- **Target:** `Churn` (1 = Yes, 0 = No).

In [ ]:
np.random.seed(42)
n_samples = 2000

# Features
contract_types = np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.5, 0.3, 0.2])
monthly_charges = np.random.normal(70, 30, n_samples)
tenure = np.random.randint(1, 72, n_samples)
total_usage = monthly_charges * tenure + np.random.normal(0, 100, n_samples)

# Churn Probability Calculation (Synthetic Logic)
churn_prob = 0.2 # Base probability
churn_prob += np.where(contract_types == 'Month-to-month', 0.4, 0.0) # High risk
churn_prob += np.where(contract_types == 'Two year', -0.15, 0.0)    # Low risk
churn_prob -= (tenure / 100) # Longer tenure -> less churn
churn_prob += (monthly_charges / 200) # High cost -> more churn

# Clip probabilities
churn_prob = np.clip(churn_prob, 0, 1)

# Generate Labels
churn = np.random.binomial(1, churn_prob)

df = pd.DataFrame({
    'Contract': contract_types,
    'MonthlyCharges': monthly_charges,
    'Tenure': tenure,
    'TotalUsage': total_usage,
    'Churn': churn
})

df.head()

## 3. Preprocessing & EDA
XGBoost handles numerical data well, but we need to encode categorical variables. We also want to check the class balance.

In [ ]:
sns.countplot(x='Churn', data=df)
plt.title('Class Distribution (0=No Churn, 1=Churn)')
plt.show()

# Encode Contract
le = LabelEncoder()
df['Contract_Encoded'] = le.fit_transform(df['Contract'])

# Prepare X and y
X = df.drop(['Churn', 'Contract'], axis=1)
y = df['Churn']

X.head()

## 4. Model Training
We use XGBClassifier. We'll use a standard train/test split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', n_estimators=100, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

## 5. Evaluation
We look at the Confusion Matrix to see false positives/negatives, and the ROC curve to judge overall discriminatory power.

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print(f"ROC AUC Score: {auc:.4f}")

In [ ]:
# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Feature Importance
plt.figure(figsize=(8, 5))
sorted_idx = model.feature_importances_.argsort()
plt.barh(X.columns[sorted_idx], model.feature_importances_[sorted_idx])
plt.xlabel("XGBoost Feature Importance")
plt.show()